# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIRˆ² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset from the Croissant URL
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display high-level metadata
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields.

Let's enumerate the available record sets and their fields.

In [ ]:
# List all record sets and their fields via their @id
record_sets = list(dataset.record_sets)
print("Available record sets:")

for rs in record_sets:
    print(f"- Record Set Name: {rs.name}")
    print(f"  Record Set @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field Name: {field.name}")
        print(f"      Field @id: {field.id}")
        print(f"      DataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` values from above.

For this example, we'll use the **main experimental results record set**. Update the code below with the appropriate `@id` for the primary record set if needed.

In [ ]:
# Gather all record set IDs (@id)
record_set_ids = [rs.id for rs in record_sets]
print("Record Set @ids:", record_set_ids)

# For this dataset, there's typically one main record set. Pick the first for demonstration.
main_record_set_id = record_set_ids[0]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns and preview the main record set
print("Fields in main record set:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes to prepare it for further analysis.

We'll:
- Filter for proteins with high abundance
- Normalize a numeric field (e.g., Abundance)
- Group by a protein classification field

**Modify `numeric_field_id` and `group_field_id` below as needed, referencing the actual `@id` fields printed above.**

In [ ]:
# Assign field @id for a numeric field for demonstration (e.g., abundance, MW, etc.)
numeric_field_id = None
candidate_numeric_fields = [c for c in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][c].dtype.kind in 'if']
print('Numeric candidate fields:', candidate_numeric_fields)

if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]  # Use the first numeric field found

if not numeric_field_id:
    raise ValueError("No numeric fields found in the main record set.")

threshold = dataframes[main_record_set_id][numeric_field_id].mean()

filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: (showing first 5)")
print(filtered_df.head())

filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records (first 5):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a categorical field
group_field_id = None
candidate_group_fields = [c for c in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][c].dtype == 'object']
print('Categorical candidate fields:', candidate_group_fields)
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]  # pick the first string field

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id} (showing first 5):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the normalized numeric field
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, kde=True)
plt.title(f"Distribution of normalized {numeric_field_id} (> mean)")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Count")
plt.show()

# Optionally, if grouping field exists, visualize mean by group
if group_field_id:
    top_groups = grouped_df.sort_values(by=numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(10,4))
    sns.barplot(y=group_field_id, x=numeric_field_id, data=top_groups)
    plt.title(f"Top 10 {group_field_id}s by Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIRˆ² dataset using `mlcroissant`, reviewed available record sets and fields via their `@id`s, filtered and normalized quantitative information, grouped proteins by categorical attributes, and visualized the results.

**Key Observations:**
- The dataset includes rich experimental protein data with multiple numeric and annotation fields.
- Using field `@id`s enables rigorous and reproducible access and analysis.
- This Croissant-based approach provides structured interoperability for downstream applications and ML pipelines.